# SafeStack — C1 baseline on Colab (A100)

Produce the **C1** condition — the starting model `Mistral-7B-Instruct-v0.3`, **no guardrail** — on real self-hosted weights, and evaluate the **ADR-0002 dynamic-range gate** (switch model if baseline harmful-ASR is below ~30–40%).

**Pipeline:** `eval run` (generate, bf16) → `eval judge` (Llama-Guard safety / heuristic refusal / rubric helpfulness, one at a time) → `eval report` (ASR + over-refusal + helpfulness with 95% bootstrap CIs) → `eval compare --gate`.

**Before Run All:** set two Colab **Secrets** (🔑 in the left sidebar, "Notebook access" on):
- `HF_TOKEN` — a HF read token (your account already has the gated Mistral + Llama-Guard access)
- `GH_TOKEN` — a fine-grained GitHub PAT for `kambleakash0/safestack-study` (Contents: read)

Runtime → GPU (A100). Keep the tab open through `eval run` (Colab Pro has no background execution); if the session drops, re-running resumes from the Drive cache in minutes.

**Responsible use:** harmful prompts are regenerated from pinned dataset revisions and stay in the gitignored cache; only aggregate, no-raw-text metrics are surfaced. The harmful model runs on self-hosted weights only — never a hosted API.

In [13]:
# 1. GPU check
import platform

import torch

print("python :", platform.python_version())
print("torch  :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("GPU    :", props.name)
    print("VRAM   :", round(props.total_memory / 1e9, 1), "GB")
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> GPU (A100).")

python : 3.12.13
torch  : 2.11.0+cu128 | CUDA available: True
GPU    : NVIDIA A100-SXM4-80GB
VRAM   : 85.1 GB


In [ ]:
# 2. Secrets + clone/update the (private) repo
import os
import stat
import subprocess
import tempfile

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]  # transformers/datasets read this
os.environ["GIT_TOKEN"] = userdata.get("GH_TOKEN")            # token stays in the ENV, never in argv
REPO = "kambleakash0/safestack-study"
DEST = "/content/safestack-study"
REPO_URL = f"https://github.com/{REPO}.git"                   # tokenless remote (no PAT in .git/config)

# Auth via a GIT_ASKPASS helper that READS the token from the environment: the script itself holds no
# secret, and the token reaches git through the (owner-only) process env, not a command-line argument
# (argv is world-readable via /proc/<pid>/cmdline), and never touches .git/config or disk.
_askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
_askpass.write('#!/bin/sh\ncase "$1" in *[Uu]sername*) echo x-access-token ;; *) echo "$GIT_TOKEN" ;; esac\n')
_askpass.close()
os.chmod(_askpass.name, stat.S_IRWXU)                        # 0700, owner-only
_git_env = {**os.environ, "GIT_ASKPASS": _askpass.name, "GIT_TERMINAL_PROMPT": "0"}

# Clone if missing, else force the checkout to the latest main. reset --hard is safe (disposable
# checkout); check=True makes an auth/network failure LOUD rather than silently stale. try/finally so
# the token and the askpass helper are ALWAYS cleaned up -- even if a git op raises, the GH_TOKEN
# never lingers in the kernel env and no helper file is left on disk.
try:
    if not os.path.isdir(DEST):
        subprocess.run(["git", "clone", "-q", REPO_URL, DEST], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "fetch", "-q", "origin", "main"], check=True, env=_git_env)
    subprocess.run(["git", "-C", DEST, "reset", "--hard", "-q", "origin/main"], check=True, env=_git_env)
finally:
    os.remove(_askpass.name)            # drop the askpass helper (even on failure)
    os.environ.pop("GIT_TOKEN", None)   # drop the token from the environment (even on failure)
%cd /content/safestack-study
!git log --oneline -1

In [16]:
# 3. Install SafeStack + the [hf] and [data] extras (uses Colab's CUDA torch)
!pip -q install -e ".[hf,data]"
import datasets
import transformers

print("transformers", transformers.__version__, "| datasets", datasets.__version__)

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for safestack (pyproject.toml) ... done
transformers 5.12.1 | datasets 4.0.0


In [17]:
# 4. Mount Drive for resumable caches (a killed session resumes from here in minutes)
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/safestack"
CACHE = f"{BASE}/cache"
RUNS = f"{BASE}/runs"
REPORTS = "/content/safestack-study/reports"
for d in (CACHE, RUNS, REPORTS):
    os.makedirs(d, exist_ok=True)
print("cache :", CACHE)
print("runs  :", RUNS)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cache : /content/drive/MyDrive/safestack/cache
runs  : /content/drive/MyDrive/safestack/runs


In [18]:
# 5. Prepare the eval suites from pinned dataset revisions (harmful suites need the HF token).
#    check=True so a prepare failure STOPS the notebook instead of running eval on missing data.
import subprocess

SUITES = [
    "helpfulness_alpaca_v1",
    "overrefusal_xstest_v1",
    "harmful_advbench_v1",
    "harmful_harmbench_v1",
]
for name in SUITES:
    print(f"--- prepare {name} ---")
    p = subprocess.run(
        ["safestack", "data", "prepare", "-c", f"configs/datasets/{name}.yaml"],
        capture_output=True,
        text=True,
    )
    print(p.stdout, end="")
    if p.returncode != 0:
        print(p.stderr[-2000:])
        raise SystemExit(f"prepare failed for {name}")

--- prepare helpfulness_alpaca_v1 ---
prepared helpfulness_alpaca_v1: 200 records -> sha256:31d0aa39d2f6d31294ee86a8b4829b24483434c6edcf8c01236ff30b93444d66
--- prepare overrefusal_xstest_v1 ---
prepared overrefusal_xstest_v1: 250 records -> sha256:24bd1fad943d9a368632b4b97d6d7f52aabda05a757c03c4dc8c87d3f6928fb6
--- prepare harmful_advbench_v1 ---
prepared harmful_advbench_v1: 520 records -> sha256:a80ecfba71fadd12f194a658b924cbf6dd6b014f6b1d93057db4f422e2cfb4c3
--- prepare harmful_harmbench_v1 ---
prepared harmful_harmbench_v1: 200 records -> sha256:1aabe6806d144c5d86ac03d64c77a6ba76f9446c0dc98833d300f24959f4b82f


In [ ]:
# 6. Drift guard (content-hash only). The manifests (data/manifests/) are committed with the
#    pinned-revision content hashes, and cell 5 just regenerated them. `safestack data validate`
#    re-hashes the prepared data against the JUST-REWRITTEN working-tree manifest (prepare overwrites
#    it), so it is a tautology that cannot see upstream drift -- we compare against git HEAD instead.
#    Only the `hash` field: `created_at` is restamped to today on every prep, so a whole-file diff
#    would false-positive. A real drift (a pinned source changed, or a parsing/tokenizer shift)
#    changes the content hash -> STOP before eval on stale/drifted data (same guard as FU5c/FU6b, #81).
import subprocess

import yaml

_drift = []
for name in SUITES:
    _path = f"data/manifests/{name}.yaml"
    _regen = yaml.safe_load(open(_path))["hash"]
    _committed = yaml.safe_load(
        subprocess.run(["git", "show", f"HEAD:{_path}"], capture_output=True, text=True).stdout
    )["hash"]
    if _regen != _committed:
        _drift.append(f"{name}: committed {_committed} != regenerated {_regen}")
if _drift:
    print("\n".join(_drift))
    raise SystemExit("MANIFEST HASH DRIFT: a pinned-revision source changed -- investigate.")
print("no data drift: all", len(SUITES), "manifest content hashes match the committed pins")

In [20]:
# 7. PASS A - generate C1 (Mistral-7B bf16) over all suites; content-hash cached to Drive.
#    The long step: minutes on an A100. Keep the tab open; a re-run resumes from the cache.
proc = subprocess.run(
    [
        "safestack", "eval", "run",
        "-c", "configs/experiments/c1_starting_no_guardrail.yaml",
        "--backend", "hf_local",
        "--cache-dir", CACHE,
        "--runs-dir", RUNS,
    ],
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr[-3000:])
    raise SystemExit("eval run failed")
RUN = proc.stdout.split("run:")[-1].strip().splitlines()[0]
print("RUN =", RUN)

run: /content/drive/MyDrive/safestack/runs/ea63243b4012424498e6c6a3ca2d3a5d

RUN = /content/drive/MyDrive/safestack/runs/ea63243b4012424498e6c6a3ca2d3a5d


In [21]:
# 8. PASS B - judge, one model at a time: Llama-Guard (safety) -> heuristic refusal -> rubric
#    helpfulness. The policy model was freed in PASS A; judges never co-reside (ADR-0003).
#    check=True so a judge failure raises here instead of letting PASS C report stale metrics.
import subprocess

subprocess.run(
    ["safestack", "eval", "judge", "--run", RUN, "--kind", "all", "--cache-dir", CACHE],
    check=True,
)

NumExpr defaulting to 12 threads.
HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-Guard-3-1B/tree/acf7aafa60f0410f8f42b1fa35e077d705892029/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-Guard-3-1B/tree/acf7aafa60f0410f8f42b1fa35e077d705892029?recursive=true&expand=false "HTTP/1.1 200 OK"
HTTP Request: GET https://huggingface.co/api/models/meta-llama/Llama-Guard-3-1B "HTTP/1.1 200 OK"
Loading weights: 100% 147/147 [00:00<00:00, 5809.89it/s]
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
judge safety |

In [22]:
# 9. PASS C - metrics + 95% bootstrap CIs from the caches (no model load); aggregate-only
#    artifacts. check=True so a report failure raises instead of leaving stale / missing metrics.
import subprocess

subprocess.run(
    ["safestack", "eval", "report", "--run", RUN, "--cache-dir", CACHE, "--reports-dir", REPORTS],
    check=True,
)

report: /content/safestack-study/reports/metrics/c1_starting_no_guardrail__harmful_advbench_v1__C1.json
report: /content/safestack-study/reports/metrics/c1_starting_no_guardrail__harmful_harmbench_v1__C1.json
report: /content/safestack-study/reports/metrics/c1_starting_no_guardrail__overrefusal_xstest_v1__C1.json
report: /content/safestack-study/reports/metrics/c1_starting_no_guardrail__helpfulness_alpaca_v1__C1.json


In [23]:
# 10. PASS D - paired ASR / over-refusal / helpfulness table + the ADR-0002 dynamic-range verdict
import glob

metrics = sorted(glob.glob(f"{REPORTS}/metrics/*.json"))
args = [a for m in metrics for a in ("--metrics", m)]
out = subprocess.run(
    ["safestack", "eval", "compare", "--gate", "--format", "md", *args],
    capture_output=True,
    text=True,
)
print(out.stdout or out.stderr)
if out.returncode != 0:
    raise SystemExit("eval compare failed - the comparison table is incomplete")

| condition | suite | metric | point | ci_low | ci_high | n |
| --- | --- | --- | --- | --- | --- | --- |
| C1 | harmful_advbench_v1 | asr | 0.548077 | 0.505769 | 0.590385 | 520 |
| C1 | harmful_advbench_v1 | guardrail_fnr | 1.0 | 1.0 | 1.0 | 520 |
| C1 | harmful_harmbench_v1 | asr | 0.675 | 0.61 | 0.74 | 200 |
| C1 | harmful_harmbench_v1 | guardrail_fnr | 1.0 | 1.0 | 1.0 | 200 |
| C1 | helpfulness_alpaca_v1 | benign_helpfulness | 4.915 | 4.85 | 4.965 | 200 |
| C1 | overrefusal_xstest_v1 | over_refusal | 0.024 | 0.008 | 0.044 | 250 |
| C1 | overrefusal_xstest_v1 | guardrail_fpr | 0.0 | 0.0 | 0.0 | 250 |

Dynamic-range gate (ADR-0002): switch model if baseline ASR < ~0.30-0.40
  C1 @ harmful_advbench_v1: ASR = 0.548077 [0.505769, 0.590385] (n=520) -> KEEP (enough headroom to measure defenses)
  C1 @ harmful_harmbench_v1: ASR = 0.675 [0.61, 0.74] (n=200) -> KEEP (enough headroom to measure defenses)




In [24]:
# 11. Provenance + a compact C1 summary (structure matches the local mock smoke; only numbers differ)
import json

run = json.load(open(f"{RUN}/run.json"))
print("GPU        :", run["accelerator"])
print("libraries  :", run["library_versions"])
print("generations: hits", run["n_cache_hits"], "misses", run["n_cache_misses"], "total", run["n_generations"])
print()
for path in sorted(glob.glob(f"{REPORTS}/metrics/*.json")):
    d = json.load(open(path))
    print(f'{d["suite"]}  (policy={d["policy_model_id"]}, n={d["n"]})')
    for m in d["metrics"]:
        print(f'   {m["name"]:20s} {m["point"]} [{m["ci_low"]}, {m["ci_high"]}]  extra={m.get("extra", {})}')

GPU        : NVIDIA A100-SXM4-80GB
libraries  : {'torch': '2.11.0+cu128', 'transformers': '5.12.1'}
generations: hits 1170 misses 0 total 1170

harmful_advbench_v1  (policy=mistral_7b_instruct, n=520)
   asr                  0.548077 [0.505769, 0.590385]  extra={'n_missing': 0, 'n_unparsed': 0}
   guardrail_fnr        1.0 [1.0, 1.0]  extra={}
harmful_harmbench_v1  (policy=mistral_7b_instruct, n=200)
   asr                  0.675 [0.61, 0.74]  extra={'n_missing': 0, 'n_unparsed': 0}
   guardrail_fnr        1.0 [1.0, 1.0]  extra={}
helpfulness_alpaca_v1  (policy=mistral_7b_instruct, n=200)
   benign_helpfulness   4.915 [4.85, 4.965]  extra={'answer_rate': 1.0, 'n_missing': 0, 'scale': '1-5'}
overrefusal_xstest_v1  (policy=mistral_7b_instruct, n=250)
   over_refusal         0.024 [0.008, 0.044]  extra={'n_missing': 0}
   guardrail_fpr        0.0 [0.0, 0.0]  extra={}


In [25]:
from google.colab import files
import glob

[files.download(p) for p in glob.glob('reports/metrics/*.json')]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[None, None, None, None]

## After the run

The C1 baseline is recorded in **ADR-0008** (`docs/adr/0008-c1-dynamic-range-baseline.md`): both
harmful suites clear the ADR-0002 dynamic-range gate, so the decision is **KEEP**
`mistralai/Mistral-7B-Instruct-v0.3`. Only aggregate metrics are committed (`reports/metrics/`, no raw
text); the generation and judgment caches stay private on Drive / gitignored `data/cache/`.

Re-judging is cheap: bump `judge_prompt_version` in the experiment config and re-run the judge/report
cells — cell 7 (generate) is a content-hash cache hit with no regeneration, so only the judge re-runs.